# Judiciary AI — Legal LLM Training Notebook
**Stack:** Python · PyMuPDF · Tesseract · spaCy · Groq · HuggingFace · QLoRA · RTX 3090/4090  
**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15  
> ⚠️ Set your `.env` file at `D:/legal_AI/.env` before running Cell 1.

In [ ]:
# ============================================================
# CELL 1 — Environment Setup
# ============================================================
import os, sys, json, re, gc, time, hashlib, logging
from pathlib import Path
from dotenv import load_dotenv

# Load env FIRST (same pattern as your main.py fix)
load_dotenv(r"D:/legal_AI/.env")

# ── Paths ─────────────────────────────────────────────────
BASE_DIR    = Path(r"D:/legal_AI")
PDF_DIR     = BASE_DIR / "data" / "raw"          # your 48,294 PDFs here
EXTRACT_DIR = BASE_DIR / "data" / "extracted"    # raw extracted text
CLEAN_DIR   = BASE_DIR / "data" / "cleaned"      # post-cleaning text
META_DIR    = BASE_DIR / "data" / "metadata"     # per-doc JSON metadata
SFT_DIR     = BASE_DIR / "data" / "datasets" / "sft"
MODEL_DIR   = BASE_DIR / "models"
CKPT_DIR    = BASE_DIR / "models" / "checkpoints"
PROD_DIR    = BASE_DIR / "models" / "production"

for d in [PDF_DIR, EXTRACT_DIR, CLEAN_DIR, META_DIR, SFT_DIR, MODEL_DIR, CKPT_DIR, PROD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logging ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(BASE_DIR / "training.log"),
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger("judiciary")

# ── GPU Check ─────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "CUDA not found — check your drivers"
GPU_NAME  = torch.cuda.get_device_name(0)
GPU_VRAM  = torch.cuda.get_device_properties(0).total_memory / 1e9
log.info(f"GPU: {GPU_NAME}  |  VRAM: {GPU_VRAM:.1f} GB")

# Auto-select batch size based on VRAM
BATCH_SIZE = 4 if GPU_VRAM >= 20 else 2   # 4090=24GB → 4, 3090=24GB → 4
GRAD_ACCUM = 8                             # effective batch = 32
MAX_SEQ_LEN = 2048

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
HF_TOKEN     = os.getenv("HF_TOKEN", "")  # optional, for gated models

log.info("✅ Cell 1 done — environment ready")

In [ ]:
# ============================================================
# CELL 2 — Install Dependencies
# ============================================================
# Run this ONCE, then comment out to speed up re-runs

import subprocess, sys

PACKAGES = [
    # PDF processing
    "pymupdf",
    "pdfplumber",
    "pdf2image",
    "pytesseract",
    # NLP
    "spacy",
    "sentence-transformers",
    # Training stack
    "transformers==4.44.0",
    "trl>=0.9.0",
    "peft>=0.12.0",
    "accelerate>=0.33.0",
    "bitsandbytes>=0.43.0",
    "datasets",
    "evaluate",
    "rouge_score",
    "bert_score",
    # Monitoring
    "wandb",
    # Utilities
    "groq",
    "python-dotenv",
    "tqdm",
    "pandas",
]

for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"⚠️  {pkg}: {result.stderr.strip()[:120]}")
    else:
        print(f"✅ {pkg}")

# Download spaCy model (transformer-based for better legal NER)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_trf"], check=True)

print("\n✅ Cell 2 done — all packages installed")

In [ ]:
# ============================================================
# CELL 3 — PDF Loader & Extraction Pipeline
# ============================================================
import fitz          # PyMuPDF
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Tesseract path (Windows) ──────────────────────────────
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# Download Tesseract from: https://github.com/UB-Mannheim/tesseract/wiki
# Also install Hindi lang pack: tessdata/hin.traineddata


def get_text_density(pdf_path: Path) -> float:
    """Measure average characters per page to decide if OCR is needed."""
    try:
        doc = fitz.open(str(pdf_path))
        total_chars = sum(len(page.get_text()) for page in doc)
        return total_chars / max(len(doc), 1)
    except Exception:
        return 0.0


def extract_digital_pdf(pdf_path: Path) -> str:
    """Extract text from native digital PDFs using pdfplumber (handles tables/columns better)."""
    text_parts = []
    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                # extract_text with tolerances handles multi-column layouts
                text = page.extract_text(x_tolerance=3, y_tolerance=3)
                if text:
                    text_parts.append(text)
    except Exception:
        # Fallback to PyMuPDF if pdfplumber fails
        try:
            doc = fitz.open(str(pdf_path))
            text_parts = [page.get_text("text") for page in doc]
        except Exception as e:
            log.warning(f"Digital extraction failed for {pdf_path.name}: {e}")
    return "\n".join(text_parts)


def preprocess_image(img: Image.Image) -> Image.Image:
    """Improve OCR accuracy: grayscale + contrast boost."""
    import PIL.ImageFilter, PIL.ImageEnhance
    img = img.convert("L")                        # grayscale
    img = PIL.ImageEnhance.Contrast(img).enhance(2.0)
    img = img.filter(PIL.ImageFilter.SHARPEN)
    return img


def extract_scanned_pdf(pdf_path: Path) -> str:
    """OCR scanned PDFs. Supports English + Hindi legal documents."""
    text_parts = []
    try:
        images = convert_from_path(
            str(pdf_path),
            dpi=300,
            poppler_path=r"C:\poppler\Library\bin"  # adjust to your poppler install
        )
        for img in images:
            img = preprocess_image(img)
            # lang="eng+hin" requires Hindi tessdata installed
            text = pytesseract.image_to_string(
                img,
                lang="eng",          # change to "eng+hin" if Hindi docs present
                config="--psm 6"     # assume uniform block of text
            )
            text_parts.append(text)
    except Exception as e:
        log.warning(f"OCR failed for {pdf_path.name}: {e}")
    return "\n".join(text_parts)


def extract_pdf(pdf_path: Path) -> dict:
    """Main extraction entry point. Auto-routes to digital or OCR path."""
    density = get_text_density(pdf_path)
    doc_type = "digital" if density > 100 else "scanned"

    if doc_type == "digital":
        text = extract_digital_pdf(pdf_path)
    else:
        text = extract_scanned_pdf(pdf_path)

    # Final fallback: if still empty, try the other method
    if len(text.strip()) < 50 and doc_type == "digital":
        log.info(f"Digital extraction empty, falling back to OCR: {pdf_path.name}")
        text = extract_scanned_pdf(pdf_path)

    return {
        "filename": pdf_path.name,
        "doc_type": doc_type,
        "text_density": density,
        "char_count": len(text),
        "text": text,
        "status": "ok" if len(text.strip()) > 50 else "empty"
    }


def batch_extract(pdf_dir: Path, out_dir: Path, max_workers: int = 4, limit: int = None):
    """Process all PDFs with a thread pool. Skips already-extracted files."""
    pdf_files = list(pdf_dir.glob("*.pdf"))
    if limit:
        pdf_files = pdf_files[:limit]

    stats = {"ok": 0, "empty": 0, "error": 0, "skipped": 0}

    def process_one(pdf_path: Path):
        out_path = out_dir / (pdf_path.stem + ".txt")
        if out_path.exists():
            return "skipped"
        try:
            result = extract_pdf(pdf_path)
            out_path.write_text(result["text"], encoding="utf-8")
            # Save mini-manifest per doc
            manifest_path = out_dir / (pdf_path.stem + ".json")
            manifest_data = {k: v for k, v in result.items() if k != "text"}
            manifest_path.write_text(json.dumps(manifest_data, indent=2), encoding="utf-8")
            return result["status"]
        except Exception as e:
            log.error(f"Failed {pdf_path.name}: {e}")
            return "error"

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one, p): p for p in pdf_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting PDFs"):
            result = future.result()
            stats[result] = stats.get(result, 0) + 1

    log.info(f"Extraction complete: {stats}")
    return stats


# ── Run (set limit=50 for testing, None for full corpus) ──
# stats = batch_extract(PDF_DIR, EXTRACT_DIR, max_workers=4, limit=50)

# Quick single-doc test
sample_pdfs = list(PDF_DIR.glob("*.pdf"))
if sample_pdfs:
    sample = extract_pdf(sample_pdfs[0])
    print(f"File    : {sample['filename']}")
    print(f"Type    : {sample['doc_type']}")
    print(f"Chars   : {sample['char_count']:,}")
    print(f"Status  : {sample['status']}")
    print(f"Preview :\n{sample['text'][:500]}")
else:
    print("⚠️  No PDFs found in PDF_DIR. Add some PDFs to D:/legal_AI/data/raw/")

log.info("✅ Cell 3 done — PDF extractor ready")

In [ ]:
# ============================================================
# CELL 4 — Data Cleaning & Metadata Extraction
# ============================================================
import spacy
import unicodedata
from dataclasses import dataclass, asdict

nlp = spacy.load("en_core_web_trf")


# ── Cleaning ──────────────────────────────────────────────

# Patterns that indicate junk lines (page numbers, headers, OCR noise)
JUNK_PATTERNS = [
    r"^\s*[-–—]?\s*\d{1,4}\s*[-–—]?\s*$",          # standalone page numbers
    r"^\s*(page|pg)\.?\s*\d+\s*$",                   # "Page 4"
    r"^\s*www\..+\.\w+\s*$",                          # URLs
    r"^\s*(IN THE|BEFORE THE|AT NEW DELHI)\s*$",      # repeated header fragments
    r"(.)(\1){5,}",                                    # OCR noise: -------, .......
    r"^\s*[_\-=\.]{4,}\s*$",                          # divider lines
]
JUNK_RE = [re.compile(p, re.IGNORECASE) for p in JUNK_PATTERNS]


def normalize_unicode(text: str) -> str:
    """Normalize unicode, fix common OCR substitutions."""
    text = unicodedata.normalize("NFKC", text)
    # Fix common OCR mistakes in legal docs
    OCR_FIXES = {
        "\u2019": "'",   # curly apostrophe
        "\u2018": "'",
        "\u201c": '"',
        "\u201d": '"',
        "l\x00": "l",    # null bytes after chars
        "\x00": "",
        "\uf0b7": "-",   # bullet OCR artifact
    }
    for bad, good in OCR_FIXES.items():
        text = text.replace(bad, good)
    return text


def is_junk_line(line: str) -> bool:
    if len(line.strip()) < 3:
        return True
    return any(pat.search(line) for pat in JUNK_RE)


def clean_text(raw: str) -> str:
    """Full cleaning pipeline."""
    text = normalize_unicode(raw)
    lines = text.split("\n")
    cleaned = []
    for line in lines:
        line = line.strip()
        if is_junk_line(line):
            continue
        # Collapse internal whitespace
        line = re.sub(r"\s{2,}", " ", line)
        cleaned.append(line)

    # Join and collapse excessive blank lines
    result = "\n".join(cleaned)
    result = re.sub(r"\n{3,}", "\n\n", result)
    return result.strip()


# ── Deduplication ─────────────────────────────────────────

def doc_fingerprint(text: str) -> str:
    """MD5 of first 2000 chars — fast near-duplicate detection."""
    return hashlib.md5(text[:2000].encode()).hexdigest()


# ── Metadata Extraction ───────────────────────────────────

@dataclass
class LegalMetadata:
    filename:    str
    case_number: str | None
    court:       str | None
    petitioner:  str | None
    respondent:  str | None
    judge:       str | None
    dates:       list
    sections:    list
    acts:        list
    doc_class:   str | None   # judgment / petition / order / notice / affidavit


# Compiled regex patterns for Indian legal docs
META_PATTERNS = {
    "case_number": re.compile(
        r"(?:WP|CRL|CS|OP|SA|FA|LPA|RFA|CM|OA|TA|MA)\s*(?:\([A-Z]+\))?\s*"
        r"(?:No\.?\s*)?\d+\s*(?:/|of)\s*\d{4}",
        re.IGNORECASE
    ),
    "court": re.compile(
        r"(?:IN THE\s+)?(?:SUPREME COURT|HIGH COURT|DISTRICT COURT|SESSIONS COURT|"
        r"NATIONAL CONSUMER|TRIBUNAL|AUTHORITY).*?(?=\n|$)",
        re.IGNORECASE
    ),
    "judge": re.compile(
        r"(?:HON'?BLE|CORAM|BEFORE)\s*:?\s*(?:MR\.?|MS\.?|MRS\.?)?\s*JUSTICE\s+([A-Z][A-Z .]+)",
        re.IGNORECASE
    ),
    "date": re.compile(
        r"(?:DATED?\s*:?\s*)?(?:\d{1,2}\s+(?:JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|"
        r"JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+\d{4}|"
        r"\d{1,2}[./\-]\d{1,2}[./\-]\d{4})",
        re.IGNORECASE
    ),
    "section": re.compile(
        r"[Ss]ection\s+\d+[A-Z]?(?:\s*(?:and|&|,|/)\s*\d+[A-Z]?)*\s*(?:of\s+the\s+[\w\s]+?Act)?",
    ),
    "act": re.compile(
        r"(?:the\s+)?(?:[A-Z][\w\s]+?Act(?:,\s*\d{4})?|I\.P\.C|C\.P\.C|Cr\.P\.C|IPC|CPC|CrPC)",
    ),
    "doc_class_keywords": {
        "judgment":  ["judgment", "judgement", "order on", "decided on", "decreed"],
        "petition":  ["petition", "petitioner", "writ petition", "PIL"],
        "order":     ["order dated", "it is ordered", "directed that"],
        "notice":    ["legal notice", "show cause notice", "notice under"],
        "affidavit": ["affidavit", "solemnly affirm", "sworn before"],
    }
}


def classify_doc(text: str) -> str:
    text_lower = text[:3000].lower()
    scores = {}
    for doc_type, keywords in META_PATTERNS["doc_class_keywords"].items():
        scores[doc_type] = sum(1 for kw in keywords if kw in text_lower)
    return max(scores, key=scores.get) if any(scores.values()) else "unknown"


def extract_metadata(filename: str, text: str) -> LegalMetadata:
    # Use first 5000 chars for speed (headers contain most metadata)
    header = text[:5000]

    # spaCy NER for persons/orgs
    doc = nlp(header)
    persons = [e.text.strip() for e in doc.ents if e.label_ == "PERSON"]
    orgs    = [e.text.strip() for e in doc.ents if e.label_ == "ORG"]

    # Regex extractions
    case_numbers = META_PATTERNS["case_number"].findall(header)
    courts       = META_PATTERNS["court"].findall(header)
    judges       = META_PATTERNS["judge"].findall(header)
    dates        = META_PATTERNS["date"].findall(text)
    sections     = META_PATTERNS["section"].findall(text)[:15]
    acts         = META_PATTERNS["act"].findall(text)[:10]

    # Heuristic: in petitioner vs respondent, look for "VERSUS" / "V/S" / "vs."
    vs_match = re.search(r"(.+?)\s+(?:VERSUS|V/S|vs?\.?)\s+(.+?)\n", header, re.IGNORECASE)
    petitioner = vs_match.group(1).strip() if vs_match else (persons[0] if persons else None)
    respondent = vs_match.group(2).strip() if vs_match else (persons[1] if len(persons) > 1 else None)

    return LegalMetadata(
        filename    = filename,
        case_number = case_numbers[0] if case_numbers else None,
        court       = courts[0].strip()[:100] if courts else (orgs[0] if orgs else None),
        petitioner  = petitioner,
        respondent  = respondent,
        judge       = judges[0].strip() if judges else None,
        dates       = list(dict.fromkeys(dates))[:5],   # unique, preserve order
        sections    = list(dict.fromkeys(sections)),
        acts        = list(dict.fromkeys(acts)),
        doc_class   = classify_doc(text)
    )


def process_extracted_docs(extract_dir: Path, clean_dir: Path, meta_dir: Path, limit: int = None):
    """Clean + extract metadata for all extracted text files."""
    txt_files = list(extract_dir.glob("*.txt"))
    if limit:
        txt_files = txt_files[:limit]

    seen_fingerprints = set()
    stats = {"cleaned": 0, "duplicate": 0, "too_short": 0}

    for txt_path in tqdm(txt_files, desc="Cleaning & extracting metadata"):
        raw = txt_path.read_text(encoding="utf-8", errors="ignore")
        cleaned = clean_text(raw)

        # Skip near-duplicates
        fp = doc_fingerprint(cleaned)
        if fp in seen_fingerprints:
            stats["duplicate"] += 1
            continue
        seen_fingerprints.add(fp)

        # Skip if too short to be useful
        if len(cleaned.split()) < 100:
            stats["too_short"] += 1
            continue

        # Save cleaned text
        (clean_dir / txt_path.name).write_text(cleaned, encoding="utf-8")

        # Extract and save metadata
        meta = extract_metadata(txt_path.stem, cleaned)
        meta_path = meta_dir / (txt_path.stem + ".json")
        meta_path.write_text(json.dumps(asdict(meta), indent=2, ensure_ascii=False), encoding="utf-8")

        stats["cleaned"] += 1

    log.info(f"Processing stats: {stats}")
    return stats


# ── Test on one doc ───────────────────────────────────────
extracted_files = list(EXTRACT_DIR.glob("*.txt"))
if extracted_files:
    sample_raw = extracted_files[0].read_text(encoding="utf-8", errors="ignore")
    sample_clean = clean_text(sample_raw)
    sample_meta  = extract_metadata(extracted_files[0].stem, sample_clean)

    print("=== CLEANED TEXT (first 300 chars) ===")
    print(sample_clean[:300])
    print("\n=== METADATA ===")
    for k, v in asdict(sample_meta).items():
        print(f"  {k:15}: {v}")
else:
    print("⚠️  No extracted .txt files found — run Cell 3 first")

# Uncomment to run full batch:
# stats = process_extracted_docs(EXTRACT_DIR, CLEAN_DIR, META_DIR, limit=100)

log.info("✅ Cell 4 done — cleaning & metadata extraction ready")

In [ ]:
# ============================================================
# CELL 5 — Dataset Generation via Groq (SFT Pairs)
# ============================================================
# Uses Llama 3.3 70B (your existing Groq setup) as teacher model
# to auto-generate instruction-response pairs from legal docs.

from groq import Groq
import random

groq_client = Groq(api_key=GROQ_API_KEY)


# ── Prompt Templates ──────────────────────────────────────

SYSTEM_PROMPT = """You are an expert Indian legal analyst. You generate high-quality 
training data for a legal AI assistant. Always respond with valid JSON only. 
No preamble, no markdown, no explanation outside the JSON."""

TASK_PROMPTS = {
    "summarize": """
Read the following Indian legal document and generate a JSON object with:
- "instruction": a lawyer's instruction to summarize this document (vary the phrasing)
- "response": a precise 3–5 sentence legal summary covering: parties, court, legal issue, holding/outcome

Document:
{text}

Return only this JSON: {{"instruction": "...", "response": "..."}}
""",

    "entity_extract": """
Read the following Indian legal document. Generate a JSON object with:
- "instruction": a request to extract legal entities from this document
- "response": a JSON string containing extracted entities: case_number, court, petitioner, 
  respondent, judge, date_of_order, sections_cited (list), acts_cited (list), doc_type

Document:
{text}

Return only this JSON: {{"instruction": "...", "response": "..."}}
""",

    "legal_qa": """
Read the following Indian legal document. Generate 3 question-answer pairs a lawyer would ask.
Focus on: legal reasoning, applicable sections, court's holding, key facts, outcome.

Document:
{text}

Return only this JSON array:
[{{"instruction": "question", "response": "detailed answer"}}, ...]
""",

    "precedent_analysis": """
Read the following Indian court judgment. Generate a JSON object with:
- "instruction": a lawyer asking about precedents or arguments in this case
- "response": analysis of key legal arguments, cited precedents, and their implications

Document:
{text}

Return only this JSON: {{"instruction": "...", "response": "..."}}
"""
}


def call_groq(prompt: str, temperature: float = 0.3, max_tokens: int = 1500) -> str:
    """Call Groq with retry on rate limit."""
    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < 2:
                wait = (attempt + 1) * 10
                log.warning(f"Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                log.error(f"Groq call failed: {e}")
                return ""


def parse_groq_response(raw: str, task: str) -> list[dict]:
    """Safely parse Groq JSON output into a list of instruction-response pairs."""
    # Strip any accidental markdown fences
    raw = re.sub(r"```json|```", "", raw).strip()
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return parsed
        elif isinstance(parsed, dict) and "instruction" in parsed:
            return [parsed]
        else:
            return []
    except json.JSONDecodeError:
        log.warning(f"JSON parse failed for task '{task}': {raw[:100]}")
        return []


def generate_sft_pairs_for_doc(text: str, filename: str) -> list[dict]:
    """Generate multiple SFT pairs from a single document using all task types."""
    # Use first 3000 chars — Groq context limit + cost control
    doc_excerpt = text[:3000]
    pairs = []

    for task_name, template in TASK_PROMPTS.items():
        prompt = template.format(text=doc_excerpt)
        raw = call_groq(prompt)
        if not raw:
            continue

        parsed = parse_groq_response(raw, task_name)
        for pair in parsed:
            if "instruction" in pair and "response" in pair:
                pair["source_file"] = filename
                pair["task_type"]   = task_name
                pairs.append(pair)

        # Small delay between tasks to respect rate limits
        time.sleep(0.5)

    return pairs


def build_sft_dataset(
    clean_dir: Path,
    sft_dir: Path,
    limit: int = 100,
    output_file: str = "train.jsonl"
):
    """Build SFT dataset from cleaned legal documents."""
    txt_files = list(clean_dir.glob("*.txt"))
    random.shuffle(txt_files)         # shuffle to get diverse sample
    txt_files = txt_files[:limit]

    out_path = sft_dir / output_file
    total_pairs = 0

    with open(out_path, "a", encoding="utf-8") as f:  # append mode = resume-safe
        for txt_path in tqdm(txt_files, desc="Generating SFT pairs"):
            text = txt_path.read_text(encoding="utf-8", errors="ignore")
            if len(text.split()) < 100:
                continue

            pairs = generate_sft_pairs_for_doc(text, txt_path.stem)
            for pair in pairs:
                f.write(json.dumps(pair, ensure_ascii=False) + "\n")
                total_pairs += 1

    log.info(f"Generated {total_pairs} SFT pairs → {out_path}")
    return total_pairs


# ── Test with one doc ─────────────────────────────────────
cleaned_files = list(CLEAN_DIR.glob("*.txt"))
if cleaned_files:
    test_text = cleaned_files[0].read_text(encoding="utf-8", errors="ignore")
    print(f"Testing dataset generation on: {cleaned_files[0].name}")
    test_pairs = generate_sft_pairs_for_doc(test_text, cleaned_files[0].stem)
    print(f"Generated {len(test_pairs)} pairs:")
    for i, pair in enumerate(test_pairs[:2], 1):
        print(f"\n--- Pair {i} [{pair.get('task_type')}] ---")
        print(f"INSTRUCTION: {pair['instruction'][:150]}")
        print(f"RESPONSE   : {pair['response'][:200]}")
else:
    print("⚠️  No cleaned .txt files found — run Cells 3 & 4 first")

# Uncomment to generate full dataset (will take hours for large corpus):
# total = build_sft_dataset(CLEAN_DIR, SFT_DIR, limit=5000)

log.info("✅ Cell 5 done — dataset generator ready")

In [ ]:
# ============================================================
# CELL 6 — Tokenizer Setup
# ============================================================
from transformers import AutoTokenizer

BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
# Alternative (no HF token needed): "unsloth/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
)

# Llama 3 doesn't have a pad token by default — fix before training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"   # required for SFT with causal LM

# Add legal domain special tokens
LEGAL_SPECIAL_TOKENS = [
    "<|judgment|>",
    "<|petition|>",
    "<|order|>",
    "<|notice|>",
    "<|affidavit|>",
    "<|section|>",
    "<|holding|>",
]
num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": LEGAL_SPECIAL_TOKENS}
)
log.info(f"Added {num_added} legal special tokens to vocabulary")

# Save tokenizer
tok_save_path = MODEL_DIR / "tokenizer"
tokenizer.save_pretrained(str(tok_save_path))

# ── Chat template formatter ────────────────────────────────
# Uses Llama 3's native chat template (already set in tokenizer config)

def format_for_sft(instruction: str, response: str, system_msg: str = None) -> str:
    """Format instruction-response pair using Llama 3's chat template."""
    if system_msg is None:
        system_msg = (
            "You are Judiciary AI, an expert Indian legal assistant. "
            "You analyze legal documents, extract entities, summarize cases, "
            "identify applicable laws, and provide accurate legal insights. "
            "Always cite specific sections and acts when relevant."
        )
    messages = [
        {"role": "system",    "content": system_msg},
        {"role": "user",      "content": instruction},
        {"role": "assistant", "content": response},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )


# ── Validate on a sample ──────────────────────────────────
sample_formatted = format_for_sft(
    instruction="Summarize this writ petition filed in Delhi High Court.",
    response="The petitioner challenges the impugned order dated 15.03.2023 on grounds of violation of Article 14."
)
tokens = tokenizer(sample_formatted, return_tensors="pt")
print(f"Vocab size      : {len(tokenizer):,}")
print(f"Sample token len: {tokens['input_ids'].shape[1]}")
print(f"\nFormatted sample:\n{sample_formatted[:500]}")

log.info("✅ Cell 6 done — tokenizer ready")

In [ ]:
# ============================================================
# CELL 7 — Model Loading with QLoRA (4-bit)
# ============================================================
# RTX 4090 (24GB): can run Llama 3.1 8B in 4-bit + LoRA comfortably
# RTX 3090 (24GB): same — this config works for both

import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)

# ── 4-bit Quantization Config ─────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,   # BF16 compute (RTX 30/40 series support this)
    bnb_4bit_use_double_quant=True,      # nested quantization saves ~0.4 bits/param extra
)

# ── Load Base Model ───────────────────────────────────────
log.info(f"Loading {BASE_MODEL_ID} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",              # auto-places layers across available GPUs
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=HF_TOKEN if HF_TOKEN else None,
    attn_implementation="flash_attention_2",   # faster attention (RTX 30/40 support)
)

# Resize embeddings for the new legal tokens we added
model.resize_token_embeddings(len(tokenizer))

# Required step before applying LoRA to a quantized model
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,   # saves VRAM at cost of ~20% speed
)
model.config.use_cache = False         # must disable with gradient checkpointing

# ── LoRA Config ───────────────────────────────────────────
# Target ALL projection layers for better legal domain adaptation
lora_config = LoraConfig(
    r=16,                  # rank — 16 is good balance for RTX 4090 VRAM
    lora_alpha=32,         # scaling factor — typically 2× rank
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj"         # FFN (MLP)
    ],
)

model = get_peft_model(model, lora_config)

# ── Stats ─────────────────────────────────────────────────
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of {total:,})")

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
print(f"VRAM allocated   : {allocated:.2f} GB")
print(f"VRAM reserved    : {reserved:.2f} GB")
print(f"VRAM headroom    : {GPU_VRAM - reserved:.2f} GB")

log.info("✅ Cell 7 done — model loaded with QLoRA")

In [ ]:
# ============================================================
# CELL 8 — Load & Prepare SFT Dataset
# ============================================================
from datasets import Dataset, load_dataset
import pandas as pd


def load_sft_jsonl(jsonl_path: Path) -> Dataset:
    """Load JSONL SFT dataset and apply chat template formatting."""
    records = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    log.info(f"Loaded {len(records)} raw SFT records from {jsonl_path.name}")

    # Format each pair using Llama 3 chat template
    formatted = []
    for rec in records:
        if "instruction" not in rec or "response" not in rec:
            continue
        text = format_for_sft(rec["instruction"], rec["response"])
        token_count = len(tokenizer(text)["input_ids"])
        # Skip samples that are too long or too short
        if 50 < token_count <= MAX_SEQ_LEN:
            formatted.append({"text": text, "token_count": token_count})

    log.info(f"After filtering: {len(formatted)} valid samples")
    return Dataset.from_list(formatted)


# ── Load dataset ──────────────────────────────────────────
sft_jsonl = SFT_DIR / "train.jsonl"

if sft_jsonl.exists() and sft_jsonl.stat().st_size > 0:
    full_dataset = load_sft_jsonl(sft_jsonl)

    # Train/val split (90/10)
    split = full_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split["train"]
    eval_dataset  = split["test"]

    print(f"Train samples : {len(train_dataset):,}")
    print(f"Eval samples  : {len(eval_dataset):,}")

    # Token length distribution
    lengths = train_dataset["token_count"]
    print(f"Token lengths → min:{min(lengths)}  median:{sorted(lengths)[len(lengths)//2]}  max:{max(lengths)}")

    print("\nSample formatted input:")
    print(train_dataset[0]["text"][:600])
else:
    log.warning(f"⚠️  {sft_jsonl} not found or empty.")
    log.warning("    Creating tiny synthetic dataset for testing purposes.")

    # Synthetic fallback for testing the training loop without real data
    SYNTHETIC_PAIRS = [
        {
            "instruction": "Summarize this writ petition.",
            "response": "The petitioner filed WP(C) 1234/2023 challenging the order dated 01.01.2023, "
                        "alleging violation of Article 14 of the Constitution. The Delhi High Court "
                        "issued notice to the respondents and granted an interim stay."
        },
        {
            "instruction": "Extract the case number and parties from this legal notice.",
            "response": '{"case_number": "CS(OS) 567/2022", "petitioner": "ABC Pvt Ltd", "\
respondent": "XYZ Corp", "court": "High Court of Delhi"}'
        },
        {
            "instruction": "What sections of law were cited in this judgment?",
            "response": "The judgment cited Section 138 of the Negotiable Instruments Act, 1881, "
                        "Section 420 IPC, and Article 21 of the Constitution of India."
        },
    ] * 100   # repeat to get 300 samples

    formatted_synthetic = [
        {"text": format_for_sft(p["instruction"], p["response"]), "token_count": 100}
        for p in SYNTHETIC_PAIRS
    ]
    full_dataset = Dataset.from_list(formatted_synthetic)
    split = full_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split["train"]
    eval_dataset  = split["test"]
    print(f"Synthetic dataset: {len(train_dataset)} train / {len(eval_dataset)} eval")

log.info("✅ Cell 8 done — dataset loaded")

In [ ]:
# ============================================================
# CELL 9 — Training Loop (SFTTrainer)
# ============================================================
import wandb
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# ── W&B (optional but recommended) ───────────────────────
WANDB_PROJECT = "judiciary-llm"
RUN_NAME      = f"sft-llama3.1-8b-{time.strftime('%Y%m%d-%H%M')}"

try:
    wandb.init(project=WANDB_PROJECT, name=RUN_NAME, config={
        "model": BASE_MODEL_ID,
        "lora_r": 16,
        "lora_alpha": 32,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "max_seq_len": MAX_SEQ_LEN,
        "train_samples": len(train_dataset),
    })
    report_to = "wandb"
    log.info(f"W&B run: {wandb.run.url}")
except Exception:
    report_to = "none"
    log.warning("W&B not configured — logging locally only")


# ── Training Config ───────────────────────────────────────
training_args = SFTConfig(
    # ── Output ────────────────────────────────────────────
    output_dir          = str(CKPT_DIR / RUN_NAME),
    run_name            = RUN_NAME,

    # ── Training Duration ─────────────────────────────────
    num_train_epochs    = 3,
    # OR use max_steps=1000 for quick experiments

    # ── Batch & Gradient ──────────────────────────────────
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,   # effective batch = 32
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},

    # ── Learning Rate ─────────────────────────────────────
    learning_rate       = 2e-4,          # standard for QLoRA SFT
    lr_scheduler_type   = "cosine",
    warmup_ratio        = 0.05,
    weight_decay        = 0.01,

    # ── Precision ─────────────────────────────────────────
    bf16                = True,          # RTX 3090/4090 both support BF16
    tf32                = True,          # extra speed on Ampere+ GPUs
    optim               = "paged_adamw_8bit",   # saves VRAM vs standard AdamW

    # ── Sequence ──────────────────────────────────────────
    max_seq_length      = MAX_SEQ_LEN,
    dataset_text_field  = "text",        # field in our Dataset that has the formatted text
    packing             = False,         # set True for ~30% speed boost if all seqs similar length

    # ── Checkpointing ─────────────────────────────────────
    save_strategy       = "steps",
    save_steps          = 200,
    save_total_limit    = 3,             # keep only last 3 checkpoints (saves disk)
    load_best_model_at_end = True,

    # ── Evaluation ────────────────────────────────────────
    eval_strategy       = "steps",
    eval_steps          = 200,

    # ── Logging ───────────────────────────────────────────
    logging_dir         = str(BASE_DIR / "logs"),
    logging_steps       = 10,
    report_to           = report_to,

    # ── Misc ──────────────────────────────────────────────
    seed                = 42,
    data_seed           = 42,
    remove_unused_columns = True,
)

# ── Trainer ───────────────────────────────────────────────
trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
)

print(f"\n{'='*50}")
print(f"Starting training run: {RUN_NAME}")
print(f"Train samples    : {len(train_dataset):,}")
print(f"Eval samples     : {len(eval_dataset):,}")
print(f"Epochs           : {training_args.num_train_epochs}")
print(f"Effective batch  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Checkpoint dir   : {training_args.output_dir}")
print(f"{'='*50}\n")

# ── Train ─────────────────────────────────────────────────
train_result = trainer.train()

# Log final metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

log.info(f"Training complete. Loss: {metrics.get('train_loss', 'N/A'):.4f}")
log.info("✅ Cell 9 done — training complete")

In [ ]:
# ============================================================
# CELL 10 — Save Adapter & Merge Model for Production
# ============================================================

ADAPTER_DIR = MODEL_DIR / "adapters" / RUN_NAME
MERGED_DIR  = PROD_DIR / RUN_NAME

# ── Save LoRA adapter (fast, ~100MB) ──────────────────────
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
log.info(f"LoRA adapter saved → {ADAPTER_DIR}")

# ── Merge adapter into base model (for deployment) ────────
# This creates a standalone model that doesn't need PEFT at inference time
# NOTE: Requires loading base model in fp16 (not 4-bit) for merging

print("Merging LoRA weights into base model (this takes ~5 minutes)...")

from peft import PeftModel

# Load fresh base model in fp16 for merging
base_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
)
base_for_merge.resize_token_embeddings(len(tokenizer))

# Load and merge LoRA weights
merged_model = PeftModel.from_pretrained(base_for_merge, str(ADAPTER_DIR))
merged_model = merged_model.merge_and_unload()   # bakes LoRA into weights

# Save merged model (will be ~16GB in fp16)
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)  # safetensors format
tokenizer.save_pretrained(str(MERGED_DIR))

# Clean up to free VRAM
del merged_model, base_for_merge
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Merged model saved → {MERGED_DIR}")
print(f"   Adapter (small): {ADAPTER_DIR}")
print(f"   Merged (deploy): {MERGED_DIR}")
log.info("✅ Cell 10 done — model saved")

In [ ]:
# ============================================================
# CELL 11 — Evaluation
# ============================================================
import evaluate
from transformers import pipeline
from peft import PeftModel

# ── Load fine-tuned model for evaluation ──────────────────
# (Load adapter on top of quantized base — avoids reloading 16GB merged model)

eval_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN if HF_TOKEN else None,
)
eval_model.resize_token_embeddings(len(tokenizer))
eval_model = PeftModel.from_pretrained(eval_model, str(ADAPTER_DIR))
eval_model.eval()

gen_pipe = pipeline(
    "text-generation",
    model=eval_model,
    tokenizer=tokenizer,
    device_map="auto",
)


def generate_response(instruction: str, max_new_tokens: int = 512) -> str:
    """Generate a response from the fine-tuned model."""
    messages = [
        {"role": "system",    "content": "You are Judiciary AI, an expert Indian legal assistant."},
        {"role": "user",      "content": instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    output = gen_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,           # greedy for eval consistency
        temperature=1.0,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0]["generated_text"]
    # Strip the prompt — return only the assistant's reply
    return generated[len(prompt):].strip()


# ── ROUGE Evaluation ──────────────────────────────────────
rouge = evaluate.load("rouge")

def evaluate_on_dataset(dataset: Dataset, n_samples: int = 50) -> dict:
    """Run ROUGE evaluation on a subset of the eval dataset."""
    sample = dataset.select(range(min(n_samples, len(dataset))))

    predictions, references = [], []

    for item in tqdm(sample, desc="Running eval"):
        # Reconstruct instruction from formatted text (between user/assistant tags)
        text = item["text"]
        # Extract instruction and reference response from the formatted text
        user_match  = re.search(r"<\|start_header_id\|>user.*?<\|end_header_id\|>\n(.+?)<\|eot_id\|>", text, re.DOTALL)
        asst_match  = re.search(r"<\|start_header_id\|>assistant.*?<\|end_header_id\|>\n(.+?)<\|eot_id\|>", text, re.DOTALL)

        if not user_match or not asst_match:
            continue

        instruction = user_match.group(1).strip()
        reference   = asst_match.group(1).strip()
        prediction  = generate_response(instruction, max_new_tokens=256)

        predictions.append(prediction)
        references.append(reference)

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    return scores


# ── Hallucination Check ───────────────────────────────────

def check_hallucination(instruction: str, source_text: str, response: str) -> dict:
    """
    Simple hallucination check: does the response contain legal citations
    (section numbers, act names) that are NOT present in the source?
    """
    # Extract section/act mentions from response
    resp_sections = set(re.findall(r"[Ss]ection\s+\d+[A-Z]?", response))
    resp_acts     = set(re.findall(r"[A-Z][\w\s]+Act,?\s*\d{4}", response))

    # Check if they appear in source
    src_lower = source_text.lower()
    hallucinated_sections = {s for s in resp_sections if s.lower() not in src_lower}
    hallucinated_acts     = {a for a in resp_acts     if a.lower() not in src_lower}

    return {
        "hallucinated_sections": list(hallucinated_sections),
        "hallucinated_acts":     list(hallucinated_acts),
        "is_clean":              len(hallucinated_sections) == 0 and len(hallucinated_acts) == 0
    }


# ── Run Evaluation ────────────────────────────────────────
print("Running ROUGE evaluation on eval set...")
rouge_scores = evaluate_on_dataset(eval_dataset, n_samples=20)

print("\n=== ROUGE Scores ===")
for metric, score in rouge_scores.items():
    print(f"  {metric:12}: {score:.4f}")

# ── Manual qualitative test ───────────────────────────────
TEST_INSTRUCTIONS = [
    "What sections of the Constitution are typically invoked in a writ petition under Article 226?",
    "Summarize the grounds for granting an interim injunction under Order XXXIX of the CPC.",
    "What is the limitation period for filing a consumer complaint under the Consumer Protection Act, 2019?",
]

print("\n=== Qualitative Responses ===")
for q in TEST_INSTRUCTIONS:
    print(f"\nQ: {q}")
    ans = generate_response(q, max_new_tokens=300)
    print(f"A: {ans[:400]}")
    print("-" * 60)

log.info("✅ Cell 11 done — evaluation complete")

In [ ]:
# ============================================================
# CELL 12 — Inference & Integration with Judiciary AI Backend
# ============================================================
# Shows how to use the fine-tuned model in your existing
# Judiciary AI FastAPI backend (replacing Groq calls)

from transformers import TextIteratorStreamer
from threading import Thread


class JudiciaryLLM:
    """
    Drop-in wrapper for your fine-tuned model.
    Mimics the Groq client interface your backend already uses.
    """

    def __init__(self, model_path: str, device: str = "auto"):
        log.info(f"Loading JudiciaryLLM from {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            device_map=device,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
            ),
        )
        self.model.eval()
        log.info("JudiciaryLLM ready")

    def generate(
        self,
        instruction: str,
        context: str = "",
        max_new_tokens: int = 512,
        temperature: float = 0.1,
        stream: bool = False,
    ) -> str:
        """
        Generate a legal response. Optionally inject RAG context.
        context = retrieved document chunks from ChromaDB/Qdrant.
        """
        system_msg = "You are Judiciary AI, an expert Indian legal assistant."
        if context:
            system_msg += (
                "\n\nUse ONLY the following retrieved context to answer. "
                "If the answer is not in the context, say so.\n\n"
                f"[CONTEXT]\n{context}\n[/CONTEXT]"
            )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": instruction},
        ]
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        gen_kwargs = dict(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else 1.0,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.pad_token_id,
        )

        if stream:
            return self._stream(gen_kwargs, inputs.shape[1])

        with torch.inference_mode():
            output_ids = self.model.generate(**gen_kwargs)

        # Decode only the new tokens (not the prompt)
        new_ids = output_ids[0][inputs.shape[1]:]
        return self.tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    def _stream(self, gen_kwargs: dict, prompt_len: int):
        """Generator that yields tokens one-by-one for streaming API responses."""
        streamer = TextIteratorStreamer(
            self.tokenizer, skip_prompt=True, skip_special_tokens=True
        )
        gen_kwargs["streamer"] = streamer
        thread = Thread(target=self.model.generate, kwargs=gen_kwargs)
        thread.start()
        for token in streamer:
            yield token
        thread.join()

    # ── Convenience methods matching your existing backend ──

    def summarize(self, document_text: str) -> str:
        return self.generate(
            instruction=f"Summarize the following Indian legal document in 4–5 sentences, "
                        f"covering: parties, court, legal issue, and outcome.\n\n{document_text[:3000]}",
            max_new_tokens=400,
        )

    def extract_entities(self, document_text: str) -> dict:
        raw = self.generate(
            instruction=f"Extract all legal entities from this document. Return ONLY valid JSON.\n\n{document_text[:3000]}",
            max_new_tokens=300,
            temperature=0,
        )
        raw = re.sub(r"```json|```", "", raw).strip()
        try:
            return json.loads(raw)
        except Exception:
            return {"raw": raw}

    def answer_question(self, question: str, context: str) -> str:
        return self.generate(
            instruction=question,
            context=context,
            max_new_tokens=512,
            temperature=0.1,
        )

    def suggest_arguments(self, document_text: str) -> str:
        return self.generate(
            instruction=(
                "Based on this legal document, suggest the strongest legal arguments "
                "for the petitioner and anticipate respondent's counter-arguments. "
                f"Reference specific sections and precedents.\n\n{document_text[:3000]}"
            ),
            max_new_tokens=600,
        )


# ── FastAPI integration snippet ───────────────────────────
# This shows how to replace your Groq client in llm_reasoning.py:

FASTAPI_INTEGRATION = '''
# In your llm_reasoning.py — replace Groq with JudiciaryLLM:

from judiciary_llm import JudiciaryLLM  # save Cell 12 class to this file

# Initialize once at startup (not per-request)
llm = JudiciaryLLM(model_path="D:/legal_AI/models/production/<RUN_NAME>")

async def analyze_document(text: str, user_id: str) -> dict:
    return {
        "summary":   llm.summarize(text),
        "entities":  llm.extract_entities(text),
        "arguments": llm.suggest_arguments(text),
    }

# Streaming endpoint:
async def stream_answer(question: str, context: str):
    for token in llm.generate(question, context=context, stream=True):
        yield token
'''
print("FastAPI integration code:")
print(FASTAPI_INTEGRATION)

# ── Quick inference test ──────────────────────────────────
# Uncomment after training is complete:

# llm = JudiciaryLLM(model_path=str(MERGED_DIR))
# test_doc = "IN THE HIGH COURT OF DELHI..." # paste a real doc snippet
# print(llm.summarize(test_doc))
# print(llm.extract_entities(test_doc))
# print(llm.answer_question("What was the outcome?", context=test_doc))

log.info("✅ Cell 12 done — inference wrapper ready")

## Quick Reference

| Cell | Purpose | Key output |
|------|---------|------------|
| 1 | Environment + GPU check | Paths, BATCH_SIZE, GROQ_API_KEY |
| 2 | Install packages | All dependencies |
| 3 | PDF extraction | `data/extracted/*.txt` |
| 4 | Clean + metadata | `data/cleaned/*.txt`, `data/metadata/*.json` |
| 5 | Dataset generation | `data/datasets/sft/train.jsonl` |
| 6 | Tokenizer | `models/tokenizer/` |
| 7 | Load model (QLoRA) | 4-bit Llama 3.1 8B in memory |
| 8 | Prepare dataset | HuggingFace Dataset objects |
| 9 | Train | `models/checkpoints/<run>/` |
| 10 | Save model | `models/adapters/` + `models/production/` |
| 11 | Evaluate | ROUGE scores + hallucination rate |
| 12 | Inference | `JudiciaryLLM` wrapper for your FastAPI backend |

### Common Issues
- **CUDA OOM**: Reduce `BATCH_SIZE` to 1 or `MAX_SEQ_LEN` to 1024
- **Tesseract not found**: Install from https://github.com/UB-Mannheim/tesseract/wiki
- **Poppler not found**: Install poppler for Windows and set `poppler_path` in Cell 3
- **HF gated model**: Get access at huggingface.co/meta-llama and set `HF_TOKEN` in `.env`
- **Flash Attention error**: Remove `attn_implementation="flash_attention_2"` from Cell 7 if it fails